# EVM CUDA Benchmark on Google Colab

This notebook builds the CUDA-accelerated Eulerian Video Magnification pipeline from scratch, profiles both the color and motion pipelines, and renders output videos. It adapts to whatever GPU Colab assigns (T4, L4, or A100).

**Before running:** set Runtime type to GPU (Runtime → Change runtime type → GPU).

The full technical writeup is in the [repository blog post](https://github.com/iamkucuk/eulerian-video-magnification-cuda/blob/main/docs/blog_speedup.md).

## 1. Environment setup

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv,noheader

In [ ]:
# Install build dependencies and Python packages
!pip install -q cmake ninja pybind11 numpy scipy opencv-python requests

In [ ]:
import os, subprocess

# Clone the repository
if not os.path.exists('evm_cuda'):
    subprocess.run(['git', 'clone', 'https://github.com/iamkucuk/eulerian-video-magnification-cuda.git'], check=True)

os.chdir('evm_cuda')

# Use the kernel-optimization branch (has the multiple-elements-per-thread optimization)
subprocess.run(['git', 'fetch', 'origin'], check=True)
subprocess.run(['git', 'checkout', 'feature/kernel-optimization'], check=False)
print(f'Repository at: {os.getcwd()}')
subprocess.run(['git', 'log', '--oneline', '-3'], check=True)

## 2. Build the CUDA extension

Auto-detects the GPU's compute capability so we compile for exactly one architecture (fast build) instead of five.

In [ ]:
import subprocess, os

# Detect the GPU's compute capability
result = subprocess.run(
    ['nvidia-smi', '--query-gpu=compute_cap', '--format=csv,noheader'],
    capture_output=True, text=True, check=True
)
compute_cap = result.stdout.strip()
# Convert "7.5" -> "75" for CMAKE_CUDA_ARCHITECTURES
cuda_arch = compute_cap.replace('.', '')
gpu_name = subprocess.run(
    ['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
    capture_output=True, text=True, check=True
).stdout.strip()
print(f'GPU: {gpu_name}, compute capability: {compute_cap}, building for sm_{cuda_arch}')

In [ ]:
# Set up pybind11 CMake dir
pybind_dir = subprocess.run(
    ['python3', '-c', 'import pybind11; print(pybind11.get_cmake_dir())'],
    capture_output=True, text=True, check=True
).stdout.strip()
os.environ['pybind11_DIR'] = pybind_dir

# Clean build
build_dir = 'cuda/build'
if os.path.exists(build_dir):
    import shutil
    shutil.rmtree(build_dir)
os.makedirs(build_dir)

# Configure: build for the detected GPU only (much faster than 5 architectures)
configure_cmd = [
    'cmake', '-S', 'cuda', '-B', build_dir,
    '-DCMAKE_BUILD_TYPE=Release',
    f'-DCMAKE_CUDA_ARCHITECTURES={cuda_arch}',
    '-G', 'Ninja'
]
print('Configuring...')
result = subprocess.run(configure_cmd, check=True, capture_output=True, text=True)
print(result.stdout[-500:] if len(result.stdout) > 500 else result.stdout)

# Build
print('Building...')
result = subprocess.run(
    ['cmake', '--build', build_dir, '--config', 'Release', '-j'],
    check=True, capture_output=True, text=True
)
# Show last few ptxas lines (register counts)
ptxas_lines = [l for l in result.stderr.split('\n') if 'ptxas' in l]
for line in ptxas_lines[-5:]:
    print(line)
print('Build complete.')

In [ ]:
# Smoke test: verify the extension loads
import sys
sys.path.insert(0, 'cuda')
from evm_cuda import _evm_cuda
print(f'CUDA extension loaded. have_cuda={_evm_cuda.have_cuda}')
print(f'binom5 filter: {_evm_cuda.binom5()}')
print(f'drop_last: {_evm_cuda.drop_last}')

## 3. Download sample videos

Fetches face.mp4 (pulse/color) and baby.mp4 (breathing/motion) from MIT CSAIL's EVM project page.

In [ ]:
import subprocess
subprocess.run(['python', 'scripts/download_samples.py', 'face', 'baby'], check=True)

import os
for f in ['data/face.mp4', 'data/baby.mp4']:
    size = os.path.getsize(f)
    print(f'{f}: {size / 1024 / 1024:.1f} MB')

## 4. Color pipeline profiler

Face pulse magnification: Gaussian downsample + ideal FFT bandpass + amplify + upsample + render.

Median of 5 iterations with a warmup run. Video decode/encode excluded.

In [ ]:
import subprocess, os
env = os.environ.copy()
env['PYTHONPATH'] = 'cuda'

result = subprocess.run(
    ['python', 'scripts/profile_color.py'],
    capture_output=True, text=True, env=env, check=True
)
print(result.stdout)

## 5. Motion pipeline profiler

Baby breathing magnification: 9-level Laplacian pyramid + IIR temporal filter + amplify + reconstruct + render.

Requires more GPU memory than color (~11 GB for baby.mp4). May fail on T4 (16 GB) if other processes are using GPU memory.

In [ ]:
import subprocess, os
env = os.environ.copy()
env['PYTHONPATH'] = 'cuda'

try:
    result = subprocess.run(
        ['python', 'scripts/profile_motion.py'],
        capture_output=True, text=True, env=env, check=True, timeout=120
    )
    print(result.stdout)
except subprocess.CalledProcessError as e:
    print(f'Motion profiler failed (exit code {e.returncode}):')
    print(e.stderr[-1000:] if e.stderr else '(no stderr)')
    print('\nThe motion pipeline needs ~11 GB GPU memory.')
    print('If you are on a T4 (16 GB), try switching to L4 or A100 in Runtime settings.')
except subprocess.TimeoutExpired:
    print('Motion profiler timed out after 120s.')

## 6. Render output videos

Runs both pipelines end-to-end (including video decode + encode) and saves the results.

In [ ]:
import sys, os, time
sys.path.insert(0, 'cuda')
os.makedirs('output', exist_ok=True)

from evm_cuda.batched import magnify_color_gdown_ideal

t0 = time.time()
magnify_color_gdown_ideal(
    'data/face.mp4', 'output/face_color.mp4',
    alpha=50, level=4, fl=50/60, fh=60/60, chrom_attenuation=1.0, sampling_rate=30.0
)
print(f'face_color.mp4 rendered in {time.time()-t0:.2f}s')

In [ ]:
from evm_cuda.batched import magnify_motion_lpyr_iir

try:
    t0 = time.time()
    magnify_motion_lpyr_iir(
        'data/baby.mp4', 'output/baby_motion.mp4',
        alpha=10, lambda_c=16, r1=0.4, r2=0.05, chrom_attenuation=0.1
    )
    print(f'baby_motion.mp4 rendered in {time.time()-t0:.2f}s')
except Exception as e:
    print(f'Motion render failed: {e}')
    print('Try L4 or A100 runtime if on T4 (memory).')

## 7. Display output videos

In [ ]:
from IPython.display import HTML, display
from base64 import b64encode
import os

def show_video(path):
    if not os.path.exists(path):
        print(f'{path} not found')
        return
    with open(path, 'rb') as f:
        data = b64encode(f.read()).decode()
    size = os.path.getsize(path) / 1024 / 1024
    display(HTML(f'<p>{path} ({size:.1f} MB)</p>'))
    display(HTML(f'<video width=480 controls><source src="data:video/mp4;base64,{data}" type="video/mp4"></video>'))

show_video('output/face_color.mp4')
show_video('output/baby_motion.mp4')

## 8. Results summary

Comparison with the H100 reference numbers from the blog post.

In [ ]:
import subprocess

# Gather GPU info
gpu_name = subprocess.run(
    ['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
    capture_output=True, text=True
).stdout.strip()
gpu_mem = subprocess.run(
    ['nvidia-smi', '--query-gpu=memory.total', '--format=csv,noheader'],
    capture_output=True, text=True
).stdout.strip()

print('=' * 60)
print('EVM CUDA Benchmark Results')
print('=' * 60)
print(f'GPU:      {gpu_name}')
print(f'Memory:   {gpu_mem}')
print()
print('Reference (NVIDIA H100):')
print('  Color:  0.081s (1.12 Gpx/s, 542 fps @ 1080p)')
print('  Motion: 0.181s (0.84 Gpx/s, 405 fps @ 1080p)')
print()
print('See the profiler output above for this GPU\'s numbers.')
print('=' * 60)